# Topological spline graph on real high-dimensional scikit-learn datasets

The reusable implementation lives in `topological_graph_embedding.visualization.workflows.high_dim`. The projection uses UMAP by default; set `REDUCER = 'pca'` for the linear PCA alternative.

In [ ]:
from pathlib import Path
import sys

working_dir = Path.cwd().resolve()
notebooks_dir = working_dir / 'notebooks' if (working_dir / 'notebooks' / '__init__.py').exists() else working_dir
project_root = notebooks_dir.parent
if not (project_root / 'topological_graph_embedding').exists():
    raise RuntimeError('Start Jupyter from the repository root or its notebooks/ directory')
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(notebooks_dir))

import matplotlib.pyplot as plt

from topological_graph_embedding.visualization.workflows.high_dim import (
    build_datasets,
    fit_datasets,
    _plot_summary,
)
from topological_graph_embedding.visualization.plots import plot_embedding_row
from topological_graph_embedding.visualization.reduction import fit_reducer

N_POINTS = 900
N_CENTROIDS = 32
SPLINE_SMOOTHING = 0.02
MAX_CYCLES = 4
REDUCER = 'pca'

datasets = build_datasets(n=N_POINTS)
models, embeddings, summary = fit_datasets(
    datasets,
    n_centroids=N_CENTROIDS,
    spline_smoothing=SPLINE_SMOOTHING,
    max_cycles=MAX_CYCLES,
)

fig, axes = plt.subplots(
    len(datasets), 4, figsize=(26, 4 * len(datasets)), squeeze=False,
)
for row, (name, (points, labels)) in enumerate(datasets.items()):
    reducer_model = fit_reducer(
        points,
        method=REDUCER,
        random_state=0,
    )
    plot_embedding_row(
        axes[row], points, labels, models[name], embeddings[name],
        projected_title=f'{name}: {REDUCER.upper()} view and spline graph',
        graph_title=f'{name}: graph embedding',
        metro_lines_title=f'{name}: metro-map lines',
        metro_points_title=f'{name}: metro-map points',
        reducer=reducer_model,
        jitter_seed=row,
        route_metrics=models[name].route_metrics_,
        metro_residual_width=0.04,
    )

fig.suptitle('High-dimensional scikit-learn datasets', fontsize=16, y=0.995)
fig.tight_layout()
figure_dir = project_root / 'notebooks' / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_dir / 'high_dim_sklearn_datasets.png', dpi=160, bbox_inches='tight')
plt.show()
_plot_summary(summary, figure_dir)

## Interactive parameter exploration

Select a dataset, adjust the fitting parameters, and press **Refit selected dataset**.

In [2]:
from topological_graph_embedding.visualization.workflows.high_dim import display_interactive_controls

display_interactive_controls(datasets)